**This notebook is used for making recommendations for a given customer**

It only makes recommendations for items not already bought by the customer and uses the trained NCF model to justify its recommendations.

In [166]:
import pandas as pd

In [167]:
interactions = pd.read_csv('data_processed/interactions.csv')

In [168]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(interactions, test_size=0.2, random_state=42, shuffle=True)

In [169]:
# All items a user bought in TRAINING set
train_user_items = train.groupby('user_id')['item_id'].apply(set).to_dict()

# All items a user bought in TEST set  
test_user_items = test.groupby('user_id')['item_id'].apply(set).to_dict()

In [170]:
import torch
def get_user_inputs(user_id, num_items, interactions, device):
    """Prepares and returns the user, item, and cluster tensors on the correct device."""
    user_tensor = torch.tensor([user_id] * num_items, dtype=torch.long).to(device)
    item_tensor = torch.tensor(range(num_items), dtype=torch.long).to(device)
    
    # Extract cluster ID (safely handles the single row extraction)
    cluster_id = interactions[interactions["user_id"] == user_id].iloc[[0]]['cluster_id'].values[0]
    cluster_tensor = torch.tensor([cluster_id] * num_items, dtype=torch.long).to(device)
    
    return user_tensor, item_tensor, cluster_tensor


def get_top_k_recommendations(user_id, num_items, interactions, train_user_items, test_user_items, model, device, top_k=10):
    """Runs prediction for a single user and checks if any test items are in the top-K."""
    # 1. Get tensors
    user_tensor, item_tensor, cluster_tensor = get_user_inputs(user_id, num_items, interactions, device)
    
    # 2. Model Inference
    predictions = model(user_tensor, item_tensor, cluster_tensor).squeeze()

    # 3. Mask known train items
    known_items = list(train_user_items.get(user_id, set()))
    predictions[known_items] = float('-inf')

    # 4. Get Top-K recommendations
    scores, idx = torch.topk(predictions, top_k)
    return scores, idx



In [ ]:
customers = test.sample(5, random_state=42)
print(customers)
num_users = train['user_id'].nunique()
num_items = train['item_id'].nunique()
print(num_users, num_items)

         user_id  item_id  purchased  cluster_id
1769168     3935     3688          0           2
1126093     2043      749          0           2
2169833     5128     3720          0           3
1840988     4155     1228          0           1
1796124     4008     1631          0           3
5878 4631


In [ ]:
import torch
from model import NCF
model = NCF(num_users, num_items, num_clusters=4)
model.load_state_dict(torch.load('model_weights/ncf_model.pth'))
model.eval()

NCF(
  (user_embedding): Embedding(5878, 64)
  (item_embedding): Embedding(4631, 64)
  (cluster_embedding): Embedding(4, 8)
  (fc1): Linear(in_features=136, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=16, bias=True)
  (output): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (sigmoid): Sigmoid()
)

In [174]:
customer1 = customers.iloc[0]
user_id = torch.tensor([customer1['user_id']])
item_id = torch.tensor([customer1['item_id']])
cluster_id = torch.tensor([customer1['cluster_id']])
user_id, item_id, cluster_id

(tensor([3935]), tensor([3688]), tensor([2]))

In [175]:
df = pd.read_csv('data_processed/cleaned_retail_data.csv')

In [176]:
def item_id_to_stock_code(item_id, df):
    """Converts a user_id back to the original customer_id using the interactions DataFrame."""
    stock_code = df[df["item_id"] == item_id].iloc[0]['StockCode']
    return stock_code

In [177]:
def get_description(item_id, df):
    return df[df["item_id"] == item_id].iloc[0]['Description']



In [178]:
from config import NUM_ITEMS, NUM_USERS
def recommend_for_user(user_id,interactions, train_user_items, test_user_items, model, df, device, num_items=NUM_ITEMS, top_k=10):
    # 1. Get top k item ids
    _, rec_ids = get_top_k_recommendations(user_id, num_items=num_items, interactions=interactions, train_user_items=train_user_items, test_user_items=test_user_items, model=model, device=device, top_k=top_k)

    # 2. For each item_id get description
    recommendations = []
    for item_id in rec_ids:
        description = get_description(item_id.item(), df)
        recommendations.append(description)
    
    return recommendations

In [179]:
recommend_for_user(
    user_id=customer1['user_id'],
    interactions=interactions,
    train_user_items=train_user_items,
    test_user_items=test_user_items,
    model=model,
    df=df,
    device='cpu',
    top_k=10
)

['BLUE HARMONICA IN BOX ',
 'PAPER CHAIN KIT VINTAGE CHRISTMAS',
 'FELTCRAFT CUSHION OWL',
 'CHARLIE + LOLA RED HOT WATER BOTTLE',
 'PLASTERS IN TIN SPACEBOY',
 'PINK BLUE FELT CRAFT TRINKET BOX',
 'PARTY BUNTING',
 'VINTAGE SNAKES & LADDERS',
 'FELTCRAFT PRINCESS CHARLOTTE DOLL',
 'BAKING SET SPACEBOY DESIGN']